In [2]:
import torch
import torch.nn.functional as F
import math
import torch.nn as nn
from torch import Tensor

**Attention**


In [3]:
def calculateAttention(query: Tensor, key: Tensor, value: Tensor) -> Tensor:
    score = torch.matmul(query, key.transpose(-2,-1))
    score = score/math.sqrt(query.shape[-1])
    score = F.softmax(score, dim=-1)
    final = torch.matmul(score, value)
    return final,score

In [4]:
def calculateMaskedAttention(query: Tensor, key: Tensor, value: Tensor, mask:Tensor) -> Tensor:
    score = torch.matmul(query, key.transpose(-2,-1))
    score = score/math.sqrt(query.shape[-1])
    if mask is not None:
        score = score.masked_fill(mask==0, float("-inf"))
    score = F.softmax(score, dim=-1)
    final = torch.matmul(score, value)
    return final,score

In [5]:
batch = 3
queries = 4
keys = 16
dimensions = 8

query = torch.rand(batch, queries, dimensions)
key = torch.rand(batch, keys, dimensions)
value = torch.rand(batch, keys, dimensions)

In [6]:
calculateAttention(query, key, value)

(tensor([[[0.4560, 0.4252, 0.5263, 0.6014, 0.3683, 0.4939, 0.4963, 0.5406],
          [0.4548, 0.4239, 0.5390, 0.6067, 0.3641, 0.4878, 0.4929, 0.5319],
          [0.4574, 0.4268, 0.5257, 0.6099, 0.3644, 0.4902, 0.4904, 0.5448],
          [0.4609, 0.4300, 0.5176, 0.6101, 0.3675, 0.4886, 0.4923, 0.5508]],
 
         [[0.4086, 0.4387, 0.4813, 0.5486, 0.6083, 0.4932, 0.5703, 0.4913],
          [0.4113, 0.4439, 0.4862, 0.5501, 0.6096, 0.4869, 0.5719, 0.4877],
          [0.4190, 0.4371, 0.4818, 0.5538, 0.6031, 0.4935, 0.5777, 0.4910],
          [0.4072, 0.4407, 0.4825, 0.5538, 0.6147, 0.4896, 0.5800, 0.4903]],
 
         [[0.5104, 0.5306, 0.4548, 0.5170, 0.5426, 0.5266, 0.5835, 0.4471],
          [0.5006, 0.5175, 0.4573, 0.5109, 0.5398, 0.5326, 0.5757, 0.4449],
          [0.4930, 0.5234, 0.4616, 0.5091, 0.5347, 0.5360, 0.5662, 0.4471],
          [0.5011, 0.5249, 0.4599, 0.5171, 0.5380, 0.5309, 0.5815, 0.4479]]]),
 tensor([[[0.0737, 0.0564, 0.0691, 0.0626, 0.0776, 0.0474, 0.0577, 0.0606,
    

In [7]:
text = "attention ! lets talk about attention"
tokens = text.split()
vocab = set(tokens)
vocab_idx_mp = { item:index for index, item in enumerate(vocab)}

In [8]:
vocab_idx_mp

{'attention': 0, 'talk': 1, 'about': 2, 'lets': 3, '!': 4}

In [9]:
inputTokens = torch.tensor([vocab_idx_mp[token] for token in tokens])

In [10]:
inputTokens

tensor([0, 4, 3, 1, 2, 0])

In [11]:
embedding_layer = nn.Embedding(num_embeddings=len(vocab), embedding_dim=8)

In [12]:
embedding_layer.weight

Parameter containing:
tensor([[ 0.1568,  0.0500,  0.2040, -2.1307,  0.7060, -0.1969, -1.2568, -0.4938],
        [-0.7606, -0.1626, -1.8295,  2.5415, -2.9517,  0.8362, -1.7860,  2.3792],
        [ 0.3574, -0.2093,  0.2726, -0.1099,  2.1879,  0.3734, -0.4824, -0.4263],
        [-1.2417,  0.9709, -0.0722, -0.2643, -1.1151, -0.3961,  0.2027, -1.5919],
        [ 1.2102,  0.6957, -0.1387,  1.7694,  0.5322, -0.3608, -0.4067,  0.7585]],
       requires_grad=True)

In [13]:
embeddings = embedding_layer(inputTokens)

In [14]:
embeddings

tensor([[ 0.1568,  0.0500,  0.2040, -2.1307,  0.7060, -0.1969, -1.2568, -0.4938],
        [ 1.2102,  0.6957, -0.1387,  1.7694,  0.5322, -0.3608, -0.4067,  0.7585],
        [-1.2417,  0.9709, -0.0722, -0.2643, -1.1151, -0.3961,  0.2027, -1.5919],
        [-0.7606, -0.1626, -1.8295,  2.5415, -2.9517,  0.8362, -1.7860,  2.3792],
        [ 0.3574, -0.2093,  0.2726, -0.1099,  2.1879,  0.3734, -0.4824, -0.4263],
        [ 0.1568,  0.0500,  0.2040, -2.1307,  0.7060, -0.1969, -1.2568, -0.4938]],
       grad_fn=<EmbeddingBackward0>)

In [15]:
query_layer = nn.Linear(in_features=8, out_features=8)
key_layer = nn.Linear(in_features=8, out_features=8)
value_layer = nn.Linear(in_features=8, out_features=8)

In [16]:
query = query_layer(embeddings)
key = key_layer(embeddings)
value = value_layer(embeddings)

In [17]:
query

tensor([[-0.2007,  0.6161, -0.6475, -0.3222, -0.5324,  0.0671,  0.4876,  0.7788],
        [-0.0941, -0.6788, -0.1043, -0.8774, -0.1441, -1.3855, -0.3285,  0.3679],
        [-0.1429,  0.7186, -0.3201,  0.2402, -0.3504,  0.4644,  0.1123, -0.6746],
        [-0.3403, -1.1785, -1.8848, -0.7754, -0.2761, -0.6551,  0.9043,  0.8690],
        [ 0.4310, -0.4357,  0.1442, -0.6927,  0.2414, -0.5261, -0.7030,  0.1691],
        [-0.2007,  0.6161, -0.6475, -0.3222, -0.5324,  0.0671,  0.4876,  0.7788]],
       grad_fn=<AddmmBackward0>)

In [18]:
attention, attentionScore = calculateAttention(query, key, value)

In [19]:
attentionScore.shape

torch.Size([6, 6])

In [20]:
right_tri_mask = torch.tril(torch.ones_like(attentionScore))

In [21]:
attention, score = calculateMaskedAttention(query, key, value, right_tri_mask)

In [22]:
score

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4708, 0.5292, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3035, 0.4080, 0.2884, 0.0000, 0.0000, 0.0000],
        [0.1026, 0.0924, 0.1172, 0.6878, 0.0000, 0.0000],
        [0.1871, 0.2347, 0.1228, 0.2974, 0.1580, 0.0000],
        [0.1743, 0.1230, 0.1718, 0.1874, 0.1693, 0.1743]],
       grad_fn=<SoftmaxBackward0>)

In [7]:
class FeedForward(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.layer1 = nn.Linear(embed_size,embed_size)
    self.layer2 = nn.Linear(embed_size,embed_size)

  def forward(self, embeddings):
    x = self.layer1(embeddings)
    x = F.gelu(x)
    x = self.layer2(x)
    return x

In [8]:
class AttentionLayer(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.embed_size = embed_size
    self.query = nn.Linear(embed_size, embed_size)
    self.key = nn.Linear(embed_size, embed_size)
    self.value = nn.Linear(embed_size, embed_size)

  def forward(self, embeddings: torch.Tensor):
    batch_size = embeddings.shape[0]
    seq_length = embeddings.shape[1]
    query = self.query(embeddings)
    key = self.key(embeddings)
    value = self.value(embeddings)
    mask = torch.tril(torch.ones(1,seq_length, seq_length))
    return calculateMaskedAttention(query,key,value,mask)

In [9]:
class TransformerBlock(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.attention = AttentionLayer(embed_size)
    self.linear = FeedForward(embed_size)
    self.layerNorm = nn.LayerNorm(embed_size)

  def forward(self, embeddings):
    context, scores = self.attention(embeddings)
    context = self.layerNorm(context)
    context = self.linear(context)
    context = F.gelu(context)
    output = context+embeddings
    return output, scores

In [10]:
class Transformer(nn.Module):
  def __init__(self, embed_size, num_layers):
    super().__init__()
    self.transformerBlocks = nn.ModuleList([TransformerBlock(embed_size) for _ in range(num_layers)])

  def forward(self, embeddings):
    attentionScores = []
    for transformerBlock in self.transformerBlocks:
      embeddings, scores = transformerBlock(embeddings)
      attentionScores.append(scores)
    return embeddings, attentionScores

In [11]:
import torch
import torch.nn as nn
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_seq_length):
        super().__init__()
        position = torch.arange(max_seq_length).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, embed_size, 2) * (-math.log(10000.0) / embed_size))

        pe = torch.zeros(max_seq_length, embed_size)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('positional_embedding', pe)

    def forward(self, x):
        return x + self.positional_embedding[:, :x.size(1), :]

In [12]:
nn.functional.embedding(torch.randint(0, 4, (1, 4)), torch.rand(4, 8))

tensor([[[0.1128, 0.1144, 0.2515, 0.6614, 0.0816, 0.0481, 0.4131, 0.5100],
         [0.2483, 0.9341, 0.5341, 0.9678, 0.3799, 0.9201, 0.4293, 0.5387],
         [0.2483, 0.9341, 0.5341, 0.9678, 0.3799, 0.9201, 0.4293, 0.5387],
         [0.7646, 0.7359, 0.2540, 0.4461, 0.9738, 0.6146, 0.2992, 0.2049]]])

In [13]:
class LanguageModel(nn.Module):
  def __init__(self, vocab_size, embed_size, num_layers):
    super().__init__()
    self.embeddingLayer = nn.Parameter(torch.rand(vocab_size, embed_size))
    self.transformer = Transformer(embed_size, num_layers)
    self.positional_encoding = SinusoidalPositionalEncoding(embed_size, vocab_size)

  def forward(self, x):
    x = nn.functional.embedding(x, self.embeddingLayer)
    x = self.positional_encoding(x)
    x,scores = self.transformer(x)
    logits = torch.matmul(x, self.embeddingLayer.T)
    return logits, scores

In [14]:
dataset = ['I love Machine Learning reading', 'Machine Leaning is fun', 'The Future is Machine Learning', 'I practice Machine Learning', 'Machine Leaning is great to learn']

In [15]:
special_tokens = ["<start>", "<end>", "<pad>"]

In [16]:
vocab = set()

In [17]:
for sentense in dataset:
  vocab.update(sentense.split())
vocab = special_tokens+list(vocab)

In [18]:
vocabToIndex = {word: index  for index,word in enumerate(vocab)}
vocab_size = len(vocab)

In [19]:
def encode(str):
  return [vocabToIndex[word] for word in str.split() ]

In [20]:
def decode(tokens):
  return " ".join([vocab[token] for token in tokens])

In [21]:
def encode_batch(strs):
  encodedSentences = [[vocabToIndex["<start>"]] + encode(str) + [vocabToIndex["<end>"]] for str in strs]
  max_len = max([len(sentence) for sentence in encodedSentences])
  encodedSentences = [encodedSentence + [vocabToIndex["<pad>"]]*(max_len-len(encodedSentence)) for encodedSentence in encodedSentences ]
  return encodedSentences

In [22]:
encodedDataSet = encode_batch(dataset)

In [23]:
encodedDataSet

[[0, 15, 4, 12, 14, 8, 1, 2],
 [0, 12, 10, 11, 16, 1, 2, 2],
 [0, 13, 9, 11, 12, 14, 1, 2],
 [0, 15, 6, 12, 14, 1, 2, 2],
 [0, 12, 10, 11, 3, 7, 5, 1]]

In [24]:
decode(encodedDataSet[0])

'<start> I love Machine Learning reading <end> <pad>'

In [25]:
encodedDataSet = torch.tensor(encodedDataSet)

In [26]:
encodedDataSet

tensor([[ 0, 15,  4, 12, 14,  8,  1,  2],
        [ 0, 12, 10, 11, 16,  1,  2,  2],
        [ 0, 13,  9, 11, 12, 14,  1,  2],
        [ 0, 15,  6, 12, 14,  1,  2,  2],
        [ 0, 12, 10, 11,  3,  7,  5,  1]])

In [27]:
input_tokens = encodedDataSet[:,:-1]

In [28]:
target_tokens = encodedDataSet[:,1:]

In [29]:
model = LanguageModel(embed_size=6, vocab_size=vocab_size, num_layers=2)

In [30]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

In [31]:
logits, scores = model(input_tokens)

In [32]:
logits[0][2].softmax(-1)

tensor([0.1984, 0.0269, 0.0804, 0.0238, 0.0645, 0.0429, 0.0640, 0.0100, 0.0152,
        0.0105, 0.0569, 0.0630, 0.0283, 0.1414, 0.0475, 0.0893, 0.0369],
       grad_fn=<SoftmaxBackward0>)

In [33]:
for v,x in zip(vocab, logits[0][2].softmax(-1)):
  print(v,x.item())

<start> 0.19841724634170532
<end> 0.02686503529548645
<pad> 0.0804453045129776
great 0.023792006075382233
love 0.06446727365255356
learn 0.04288676381111145
practice 0.0639638751745224
to 0.010005205869674683
reading 0.015227292664349079
Future 0.01047517079859972
Leaning 0.056941207498311996
is 0.06304947286844254
Machine 0.028263308107852936
The 0.14143262803554535
Learning 0.04750208184123039
I 0.08932902663946152
fun 0.036937084048986435


In [34]:
logits.view(-1, logits.shape[-1]).shape

torch.Size([35, 17])

In [48]:
target_tokens.reshape(-1)

tensor([15,  4, 12, 14,  8,  1,  2, 12, 10, 11, 16,  1,  2,  2, 13,  9, 11, 12,
        14,  1,  2, 15,  6, 12, 14,  1,  2,  2, 12, 10, 11,  3,  7,  5,  1])

In [49]:
loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), target_tokens.reshape(-1))

In [50]:
loss

tensor(3.3096, grad_fn=<NllLossBackward0>)

In [53]:
num_epochs = 500
for i in range(num_epochs):
  optimizer.zero_grad()
  logits,score = model(input_tokens)
  loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), target_tokens.reshape(-1))
  loss.backward()
  optimizer.step()
  print(f"epoch: {i}, loss:{loss.item()}")

epoch: 0, loss:0.9587673544883728
epoch: 1, loss:0.9489964246749878
epoch: 2, loss:0.9449169635772705
epoch: 3, loss:0.9352409243583679
epoch: 4, loss:0.9280779361724854
epoch: 5, loss:0.9223818182945251
epoch: 6, loss:0.9130451679229736
epoch: 7, loss:0.9078798294067383
epoch: 8, loss:0.9000895619392395
epoch: 9, loss:0.892792820930481
epoch: 10, loss:0.8868733048439026
epoch: 11, loss:0.8807490468025208
epoch: 12, loss:0.8739868998527527
epoch: 13, loss:0.8670182228088379
epoch: 14, loss:0.8604810833930969
epoch: 15, loss:0.8544952869415283
epoch: 16, loss:0.8501309156417847
epoch: 17, loss:0.8536149263381958
epoch: 18, loss:0.8949787616729736
epoch: 19, loss:0.971858561038971
epoch: 20, loss:0.8407983779907227
epoch: 21, loss:0.8530704975128174
epoch: 22, loss:0.8463431000709534
epoch: 23, loss:0.8146166205406189
epoch: 24, loss:0.8237061500549316
epoch: 25, loss:0.8228480815887451
epoch: 26, loss:0.8154062628746033
epoch: 27, loss:0.807296097278595
epoch: 28, loss:0.793672204017639

In [86]:
logits.argmax(dim=-1)

tensor([[12]])

In [66]:
op = logits.argmax(dim = -1).numpy()

In [68]:
import numpy as np
v_lookup = np.vectorize(lambda x: vocab[x])

In [70]:
v_lookup(op)

array([['Machine', 'love', 'Machine', 'Learning', 'reading', '<end>',
        '<pad>'],
       ['Machine', 'Leaning', 'is', 'fun', '<end>', '<pad>', '<pad>'],
       ['Machine', 'Future', 'is', 'Machine', 'Learning', '<end>',
        '<pad>'],
       ['Machine', 'love', 'Machine', 'Learning', '<end>', '<pad>',
        '<pad>'],
       ['Machine', 'Leaning', 'is', 'fun', 'to', 'learn', '<end>']],
      dtype='<U8')

In [87]:
input = "<start>"
inputtokens = torch.tensor(encode(input)).unsqueeze(0)
logits,score = model(inputtokens)
last_token_pred = logits[0,-1:].softmax(dim=-1)
last_token_logits = last_token_pred.argmax(dim=-1,keepdim=True)
new_seq = torch.cat([inputtokens,last_token_logits], dim=1)

In [91]:
decode(new_seq[0].tolist())

'<start> Machine'

In [95]:
torch.tensor(encode(input))

tensor([0])

In [104]:
def generation(prefix, max_len=15):
  inputtokens = torch.tensor(encode(input)+encode(prefix)).unsqueeze(0)
  for _ in range(max_len):
    with torch.no_grad():
      logits,score = model(inputtokens)
      logit_pred = logits[0, -1:].softmax(dim=-1)
      token = logit_pred.argmax(dim=-1, keepdim=True)
      inputtokens = torch.cat([inputtokens, token], dim=1)

    if inputtokens[0][-1] == vocabToIndex["<end>"]:
      break
  return decode(inputtokens[0].tolist())

In [105]:
generation("Machine")

'<start> Machine Leaning is fun <end>'